# 🎬 AI Text-to-Video Generator — Wan 2.1

Google Colab notebook: prompt likho, **Generate Video** dabao, aur MP4 pao.

**Recommended runtime:** Runtime → Change runtime type → T4 GPU (agar available ho).

> Note: Yeh pretrained open-source model ko run karta hai; Colab mein model ko scratch se train nahi karta. Free GPU availability/limits Google ke control mein hain. Pehli baar model weights download honge (kaafi GB), aur generation mein time lag sakta hai.

In [ ]:
# 1) Install dependencies
!pip -q install -U diffusers transformers accelerate ftfy gradio imageio-ffmpeg

In [ ]:
# 2) Check GPU
import torch, gc
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU nahi mila. Colab mein Runtime > Change runtime type > GPU select karke dobara run karein.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 3) Load Wan 2.1 model (first run downloads model weights)
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

# T4 GPUs use float16; newer GPUs can use bfloat16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("Using dtype:", DTYPE)

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32
)

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=DTYPE
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=3.0
)

# Saves GPU memory by moving model components between CPU and GPU.
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

print("Model ready!")

In [ ]:
# 4) Video generation function
import os, uuid

NEGATIVE_PROMPT = (
    "blurry, low quality, low resolution, flicker, jitter, distorted objects, "
    "deformed anatomy, extra limbs, bad hands, text, subtitles, watermark, "
    "static image, oversaturated"
)

def generate_video(prompt, steps=25, seed=42):
    prompt = (prompt or "").strip()
    if len(prompt) < 8:
        raise ValueError("Please thoda detailed prompt likhein (at least 8 characters).")

    generator = torch.Generator(device="cuda").manual_seed(int(seed))

    # 49 frames at 16 FPS ≈ 3 seconds. Dimensions are set for 480p landscape.
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=480,
        width=832,
        num_frames=49,
        num_inference_steps=int(steps),
        guidance_scale=5.0,
        generator=generator
    )

    frames = result.frames[0]
    os.makedirs("/content/generated_videos", exist_ok=True)
    output_path = f"/content/generated_videos/video_{uuid.uuid4().hex[:8]}.mp4"
    export_to_video(frames, output_path, fps=16)

    del result, frames
    gc.collect()
    torch.cuda.empty_cache()
    return output_path

print("Generator function ready.")

## 5) Apna prompt likho aur video banao

Prompt mein subject, location, action, lighting aur camera movement describe karo. English prompts aam taur par achhe results dete hain.

In [ ]:
# 6) Launch simple web app
import gradio as gr

demo = gr.Interface(
    fn=generate_video,
    inputs=[
        gr.Textbox(
            label="Describe your video",
            placeholder="A cinematic shot of a white sports car driving through a neon-lit Tokyo street at night, rain reflections, smooth tracking camera, realistic lighting...",
            lines=4
        ),
        gr.Slider(15, 35, value=25, step=5, label="Quality / inference steps (higher = slower)"),
        gr.Number(value=42, precision=0, label="Random seed")
    ],
    outputs=gr.Video(label="Your generated MP4 video"),
    title="🎬 Wan 2.1 — AI Text to Video",
    description="Prompt likho aur Generate dabao. Generation ko complete hone mein kuch waqt lag sakta hai."
)

demo.queue().launch(share=True, debug=True)

## Tips
- Specific prompts likho: subject + action + setting + lighting + camera movement.
- Agar CUDA out-of-memory aaye, runtime restart karke dobara run karo; doosre apps/tabs mein GPU usage band rakho.
- 25–35 inference steps zyada waqt le sakte hain. Free Colab GPU par speed aur availability guaranteed nahi hoti.
- Output video Gradio player se preview/download ki ja sakti hai.

**Example prompt:** `A majestic tiger walking slowly through a misty forest at sunrise, golden rays passing through trees, cinematic tracking shot, realistic fur, natural motion, 4K film look.`